# Generate chunk embeddings

Create structure-aware article chunks, then generate chunk embeddings for all configured models.

In [ ]:
import gc
import json
import logging
from pathlib import Path

import numpy as np

logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'search' / 'articles_normalized.jsonl'
CHUNK_PATH = DATA_DIR / 'processed' / 'search' / 'article_chunks.jsonl'
OUTPUT_DIR = DATA_DIR / 'processed' / 'embeddings'
MAX_CHARS, OVERLAP_CHARS = 1200, 150
MODELS = {
    'sealion': {'model': 'aisingapore/SEA-LION-E5-Embedding-600M', 'provider': 'sentence-transformer', 'batch_size': 8, 'output': 'chunks_sealion_e5_600m'},
    'granite-r2': {'model': 'ibm-granite/granite-embedding-311m-multilingual-r2', 'provider': 'sentence-transformer', 'batch_size': 2, 'output': 'chunks_granite_311m_r2'},
}

def split_article(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    blocks, current = [], ''
    for line in lines:
        candidate = f'{current}\n{line}' if current else line
        if len(candidate) <= MAX_CHARS:
            current = candidate
        else:
            if current: blocks.append(current)
            current = line
    if current: blocks.append(current)
    chunks = []
    for block in blocks:
        if len(block) <= MAX_CHARS:
            chunks.append(block); continue
        start = 0
        while start < len(block):
            end = min(start + MAX_CHARS, len(block))
            chunks.append(block[start:end])
            if end == len(block): break
            start = end - OVERLAP_CHARS
    return chunks

def create_chunks():
    records = [json.loads(line) for line in INPUT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    output = []
    for article in records:
        chunks = split_article(article.get('search_text', article.get('text', '')))
        for index, text in enumerate(chunks, start=1):
            output.append({'chunk_id': f"{article['id']}-chunk-{index}", 'article_id': article['article_id'], 'document_id': article['document_id'], 'article_number': article['article_number'], 'chunk_index': index, 'chunk_count': len(chunks), 'text': text, 'search_text': text, 'document_title': article.get('document_title'), 'document_number': article.get('document_number'), 'document_type': article.get('document_type'), 'source_url': article.get('source_url'), 'pdf_url': article.get('pdf_url'), 'chapter_number': article.get('chapter_number'), 'chapter_title': article.get('chapter_title'), 'section_number': article.get('section_number'), 'section_title': article.get('section_title')})
    CHUNK_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHUNK_PATH.write_text(''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in output), encoding='utf-8')
    return output

def batches(values, size):
    for start in range(0, len(values), size): yield values[start:start + size]

def normalize_vectors(vectors):
    vectors = np.asarray(vectors, dtype='float32')
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

def encode_huggingface(records, model_name, batch_size):
    import torch
    from transformers import AutoModel, AutoTokenizer
    tokenizer, model = AutoTokenizer.from_pretrained(model_name), AutoModel.from_pretrained(model_name)
    model.eval(); vectors = []
    with torch.no_grad():
        for batch in batches([r['search_text'] for r in records], batch_size):
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
            output = model(**encoded).last_hidden_state
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            vectors.append(((output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)).cpu().numpy())
    return normalize_vectors(np.concatenate(vectors, axis=0))

def encode_sentence_transformer(records, model_name, batch_size):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    vectors = model.encode([r['search_text'] for r in records], batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    del model; gc.collect()
    return np.asarray(vectors, dtype='float32')

def write_outputs(records, vectors, config):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True); prefix = config['output']
    np.save(OUTPUT_DIR / f'{prefix}.npy', vectors)
    metadata = OUTPUT_DIR / f'{prefix}_metadata.jsonl'
    metadata.write_text(''.join(json.dumps({k: v for k, v in r.items() if k != 'search_text'}, ensure_ascii=False) + '\n' for r in records), encoding='utf-8')
    manifest = {'model': config['model'], 'provider': config['provider'], 'input_path': str(CHUNK_PATH), 'chunk_count': len(records), 'embedding_dimension': int(vectors.shape[1]), 'normalized_embeddings': True}
    (OUTPUT_DIR / f'{prefix}_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('Chunk input:', CHUNK_PATH)

In [ ]:
records = create_chunks()
results = {}
for model_key, config in MODELS.items():
    if config['provider'] == 'huggingface-mean-pooling':
        vectors = encode_huggingface(records, config['model'], config['batch_size'])
    else:
        vectors = encode_sentence_transformer(records, config['model'], config['batch_size'])
    write_outputs(records, vectors, config)
    results[model_key] = {'chunks': len(records), 'shape': list(vectors.shape)}
    del vectors; gc.collect()
print(json.dumps(results, ensure_ascii=False, indent=2))